# Multi-objective Bayesian Optimization Tutorial 1: Acquisition functions

All source code lives in `../src/`. Make sure your kernel uses the `uv` environment:

```bash
cd tutorial
uv run jupyter lab
```

In [ ]:
import sys, pathlib

# Add src/ to the path so imports work without installing the package.
src = pathlib.Path("../src").resolve()
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

import torch
import matplotlib.pyplot as plt


print("torch", torch.__version__)
print("src path:", src)

## 1. D=1, M=2 objective functions

Let's look at 1D functions for visualization.

In [ ]:
from hypervolume import _nondominated
from test_functions import get_test_function

# Plot Sinusoidal objectives across its 1D input domain
fn = get_test_function("Sinusoidal")

bounds = fn.search_space.bounds  # shape: (2, d)
x = torch.linspace(bounds[0, 0], bounds[1, 0], 400).unsqueeze(1)
y = fn(x)

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
base_colors = ["#e66739", "#1d3557"]

# Non-dominated points
y_pareto, nondom_idx = _nondominated(y)

# Objectives vs input
axes[0].plot(x[:, 0].numpy(), y[:, 0].numpy(), label="f1(x)", linewidth=2, color=base_colors[0])
axes[0].scatter(x[nondom_idx, 0].numpy(), y[nondom_idx, 0].numpy(), color=base_colors[0])
axes[0].plot(x[:, 0].numpy(), y[:, 1].numpy(), label="f2(x)", linewidth=2, color=base_colors[1])
axes[0].scatter(x[nondom_idx, 0].numpy(), y[nondom_idx, 1].numpy(), color=base_colors[1])
axes[0].set_xlabel("x")
axes[0].set_ylabel("objective value")
axes[0].set_title("Objectives vs input")
axes[0].legend()

# Objective-space curve
axes[1].plot(y[:, 0].numpy(), y[:, 1].numpy(), color="#50bbb2", linewidth=2)
axes[1].scatter(y_pareto[:, 0].numpy(), y_pareto[:, 1].numpy(), color="#0f766e")
axes[1].set_xlabel("f1(x)")
axes[1].set_ylabel("f2(x)")
axes[1].set_title("Objective space")

fig.tight_layout()
plt.show()

## 2. Acquisition functions

Fit a GP model on a few initial observations and look at the acquisition functions.

In [ ]:
from surrogate import fit_gp, predict, LogNormalPrior
from acq import get_acquisition, REGISTRY as ACQ_REGISTRY

# Collect 3 observations from fnaffer1
X_obs = torch.tensor([-1.5, -1.0, 0.5, 3.3]).unsqueeze(1)
Y_obs = fn(X_obs)

# Standardise observations for GP
Y_obs_mean, Y_obs_std = Y_obs.mean(0), Y_obs.std(0).clamp(min=1e-6)
Y_obs_norm = (Y_obs - Y_obs_mean) / Y_obs_std

# Train GPs on observations
gp_models = fit_gp(X_obs, Y_obs_norm, n_iter=100, min_noise=1e-5, lengthscale_prior=LogNormalPrior(0.0, 0.5))

# Make predictions on the grid
mu_grid, sigma_grid = predict(gp_models, x)

# Prepare for acquisition function evaluation
pareto_y_obs, nondom_idx = _nondominated(Y_obs_norm)
ref_norm_obs = (fn.ref_point - Y_obs_mean) / Y_obs_std

# Create figure with 2 subplots
fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)

# Subplot 1: Objectives with GP surrogates
for obj_idx in range(2):
    mu_obj = mu_grid[:, obj_idx].detach().numpy() * Y_obs_std[obj_idx].item() + Y_obs_mean[obj_idx].item()
    sigma_obj = sigma_grid[:, obj_idx].detach().numpy()
    y_obj = y[:, obj_idx].numpy()
    
    axes[0].plot(x[:, 0].numpy(), y_obj, label=f"f{obj_idx+1}(x)", linewidth=1, color=base_colors[obj_idx])
    axes[0].plot(x[:, 0].numpy(), mu_obj, "--", linewidth=2, alpha=0.8, color=base_colors[obj_idx])
    axes[0].fill_between(
        x[:, 0].numpy(), 
        mu_obj - 1.96 * sigma_obj, 
        mu_obj + 1.96 * sigma_obj, 
        alpha=0.2, color=base_colors[obj_idx]
    )
    axes[0].scatter(X_obs[:, 0].numpy(), Y_obs[:, obj_idx].numpy(), s=100, zorder=5, marker='.', color=base_colors[obj_idx])

axes[0].set_ylabel("objective value")
axes[0].legend()

# Subplot 2: Acquisition functions
x_cands_acq = torch.linspace(fn.search_space.bounds[0, 0], fn.search_space.bounds[1, 0], 150).unsqueeze(1)

# Create a second y-axis for acquisition functions
acq_names = list(ACQ_REGISTRY.keys())  # or your custom list/order


acq_colors = {acq_name: color for acq_name, color in zip(acq_names, ["#92b82b", "#420f55", "#50bbb2"])}

# One axis per acquisition
for i, acq_name in enumerate(acq_names):
    if i == 0:
        ax = axes[1]
        axes[1].set_xlabel("x")

    else:
        ax = axes[1].twinx()

    # Color y-ticks/spines by acquisition function and hide numeric tick labels
    c = acq_colors.get(acq_name, "black")
    ax.spines["right"].set_color(c)
    ax.tick_params(axis="y", colors=c, labelleft=False, labelright=False)

    # Remove numeric tick labels on x/y for all currently used axes
    ax.tick_params(axis="y", labelleft=False, labelright=False)


    # Evaluate acquisition function on the grid
    if acq_name == "EHVI":
        acq = get_acquisition(acq_name, models=gp_models, pareto_y=pareto_y_obs, ref_point=ref_norm_obs)
    elif acq_name == "MESMO":
        bounds_norm_obs = torch.stack([
            (torch.zeros(fn.d) - X_obs.mean(0)) / X_obs.std(0).clamp(min=1e-6),
            (torch.ones(fn.d)  - X_obs.mean(0)) / X_obs.std(0).clamp(min=1e-6),
        ])
        acq = get_acquisition(acq_name, models=gp_models, bounds=bounds_norm_obs, n_grid=32)
    elif acq_name == "ParEGO":
        acq = get_acquisition(acq_name, models=gp_models, train_y=Y_obs_norm)
    
    acq_vals = acq(x_cands_acq).detach().numpy()

    # Maximum of acquisition function
    acq_max_idx = acq_vals.argmax()

    ax.plot(x_cands_acq[:, 0].numpy(), acq_vals, label=acq_name, linewidth=2, color=acq_colors[acq_name])
    ax.axvline(x=x_cands_acq[acq_max_idx, 0].item(), color=acq_colors[acq_name], linestyle="--", alpha=0.8)

    axes[0].axvline(x=x_cands_acq[acq_max_idx, 0].item(), color=acq_colors[acq_name], linestyle="--", alpha=0.8)


    

handles, labels = [], []
for a in fig.axes:
    h, l = a.get_legend_handles_labels()
    for hh, ll in zip(h, l):
        if ll in acq_names and ll not in labels:
            handles.append(hh)
            labels.append(ll)
axes[1].legend(handles, labels, loc="upper right")


fig.tight_layout()
plt.show()